In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [3]:
df = pd.read_csv("../data/raw/hotel_booking.csv")

print("Dataset Shape:", df.shape)

df.head()


Dataset Shape: (119390, 36)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,name,email,phone-number,credit_card
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,Ernest Barnes,Ernest.Barnes31@outlook.com,669-792-1661,************4322
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,Andrea Baker,Andrea_Baker94@aol.com,858-637-6955,************9157
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,Rebecca Parker,Rebecca_Parker@comcast.net,652-885-2745,************3734
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,Laura Murray,Laura_M@gmail.com,364-656-8427,************5677
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,Linda Hines,LHines@verizon.com,713-226-5883,************5498


In [5]:
# Missing value Handling

In [6]:
missing = df.isnull().sum()

missing[missing > 0]


children         4
country        488
agent        16340
company     112593
dtype: int64

In [7]:
# Country
df["country"] = df["country"].fillna(
    df["country"].mode()[0]
)


In [8]:
# Children
df["children"] = df["children"].fillna(
    df["children"].median()
)


In [9]:
df.drop("company", axis=1, inplace=True)

In [10]:
print("Dataset Shape:", df.shape)

df.head()


Dataset Shape: (119390, 35)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,name,email,phone-number,credit_card
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,Ernest Barnes,Ernest.Barnes31@outlook.com,669-792-1661,************4322
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,Andrea Baker,Andrea_Baker94@aol.com,858-637-6955,************9157
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,Rebecca Parker,Rebecca_Parker@comcast.net,652-885-2745,************3734
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,Laura Murray,Laura_M@gmail.com,364-656-8427,************5677
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,Linda Hines,LHines@verizon.com,713-226-5883,************5498


In [11]:
df["agent"] = df["agent"].fillna(0)

In [12]:
df.isnull().sum().sum()


np.int64(0)

### Missing Values were handled using

- Mode for country column
- Median for children column
- Zero Replacement for agent column
- Remove company column


In [13]:
# Handle Duplicate Records

In [14]:
duplicates = df.duplicated().sum()

print("Duplicate Records:", duplicates)

Duplicate Records: 0


In [15]:
df = df.drop_duplicates()

print("New Shape:", df.shape)

New Shape: (119390, 35)


Duplicate booking records were identified and removed to prevent bias during model training.

In [17]:
# Remove Unnecessary Fields


In [18]:
privacy_cols = [
    "name",
    "email",
    "phone-number",
    "credit_card"
]


In [19]:
leakage_cols = [
    "reservation_status",
    "reservation_status_date"
]

In [20]:
df.drop(
    columns=privacy_cols + leakage_cols,
    inplace=True
)

print(df.shape)



(119390, 29)


Personal identification fields and data leakage fields were removed.


In [24]:
# Feature Engineering

In [23]:

df["total_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)


df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)


In [25]:
df[["total_nights","total_guests"]].head()

,total_nights,total_guests
0,0,2.0
1,0,2.0
2,1,1.0
3,1,1.0
4,2,2.0


In [26]:
# save cleaned dataset

In [32]:
os.makedirs(
    "../data/processed",
    exist_ok=True
)

df.to_csv(
    "../data/processed/cleaned_dataset.csv",
    index=False
)

print("Dataset Saved Successfully")


Dataset Saved Successfully


In [28]:
# Train and Test split

In [30]:
X = df.drop("is_canceled", axis=1)
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [31]:
print(X_train.shape)
print(X_test.shape)


(95512, 30)
(23878, 30)


An 80:20 train-test split was used.
Stratified sampling preserved cancellation class distribution.


In [33]:
# Categorical Encoding

In [34]:
cat_cols = X_train.select_dtypes(
    include=["object"]
).columns

print(cat_cols)


Index(['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment',
       'distribution_channel', 'reserved_room_type', 'assigned_room_type',
       'deposit_type', 'customer_type'],
      dtype='str')


C:\Users\user\AppData\Local\Temp\ipykernel_7420\3321765439.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(


In [43]:
cat_cols = X_train.select_dtypes(
    include=["object"]
).columns

print(cat_cols)


Index([], dtype='str')


In [ ]:
X_train = pd.get_dummies(
    X_train,
    columns=cat_cols,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=cat_cols,
    drop_first=True
)


In [44]:
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)


In [45]:
# Feature Scaling

In [46]:
num_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

In [47]:
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(
    X_train[num_cols]
)

X_test[num_cols] = scaler.transform(
    X_test[num_cols]
)


In [48]:
X_train.head()


,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,...,assigned_room_type_H,assigned_room_type_I,assigned_room_type_K,assigned_room_type_L,assigned_room_type_P,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
105447,-0.784882,1.193298,-1.552382,-1.000803,-0.928908,-0.261137,-1.464604,-0.259968,-0.078866,-0.180512,...,False,False,False,False,False,False,False,False,True,False
85242,-0.897168,-0.221372,-1.185007,-0.773337,-0.928908,0.262232,0.246367,-0.259968,-0.078866,-0.180512,...,False,False,False,False,False,False,False,False,False,True
65604,-0.101806,1.193298,-0.964581,-1.114537,-0.928908,0.262232,0.246367,-0.259968,-0.078866,-0.180512,...,False,False,False,False,False,True,False,False,True,False
17345,-0.129877,-1.636041,0.872297,0.818927,3.077406,3.925816,0.246367,-0.259968,-0.078866,-0.180512,...,False,False,False,False,False,False,False,False,False,False
117786,-0.897168,1.193298,0.357971,-0.773337,-0.928908,-0.784506,-1.464604,-0.259968,-0.078866,-0.180512,...,False,False,False,False,False,False,False,False,True,False


Standard Scaler was used to normalized numerical variables.
The scaler was fitted only on training data and then applied to the testing data.

## Data Leakage Prevention Strategy

The following measures were implemented:

1. reservation_status removed
2. reservation_status_date removed
3. Train-test split performed before scaling
4. Encoding performed after train-test split
5. StandardScaler fitted only on training data
6. Test data was never used during training


In [49]:
os.makedirs(
    "../data/processed",
    exist_ok=True
)

df.to_csv(
    "../data/processed/cleaned_dataset_2.csv",
    index=False
)

print("Dataset Saved Successfully")


Dataset Saved Successfully
